In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))


# 문장 -> 벡터(1차원 숫자 배열 [8.1,9.1,2,5,4, ....])
- openAI API : https://platform.openai.com/ 의 키를 .env에 등록  => OPENAI_API_KEY
- upstage : 우리나라 회사. https://console.upstage.ai/api-keys 의 키를 .env에 등록 -> UPSTAGE_KEY

# 1. 환경변수 로드

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

# 2. 유사도 계산 함수
- https://www.pinecone.io/learn/vector-similarity/
    1. 유클리드 거리 : 두 벡터간의 거리가 유사한지
    2. 코사인 유사도 : 두 벡터간의 방향이 유사한지
    3. dot product : 두 벡터간의 곱을 사용하여 거리와 방향을 모두 고려

In [3]:
import numpy as np
def cosine_similarity(vec1, vec2):
    """두 백터 사이의 코사인 유사도 계산"""
    dot_product = np.dot(vec1, vec2)   # 행렬의 곱을 계산
    norm_vec1 = np.linalg.norm(vec1)   # vec1의 거리(길이) 계산
    norm_vec2 = np.linalg.norm(vec2)   # vec2의 거리(길이) 계산
    if norm_vec1==0 or norm_vec2==0:
        return 0.0
    return dot_product / (norm_vec1*norm_vec2)  #

# 3. openAI API의 embedding model 사용

In [4]:
from openai import OpenAI
oepnai_client = OpenAI()

In [18]:
# text-embedding-3-large
response = oepnai_client.embeddings.create(
    input="king",
    model="text-embedding-3-large"
)

In [19]:
# 문장을 벡터로 변경
import numpy as np
king_vectoer = np.array(response.data[0].embedding)
# 3072 개 행의 1차원 배열로 변경
print(king_vectoer.shape)
print(king_vectoer)

(3072,)
[ 0.01040417  0.02499519 -0.0014776  ...  0.00835009  0.01049861
 -0.00254005]


In [15]:
queen_response = oepnai_client.embeddings.create(
    input="queen",
    model="text-embedding-3-large"
)

In [16]:
queen_vector = np.array(queen_response.data[0].embedding)
print(queen_vector.shape)
print(queen_vector)

(3072,)
[-0.01385735  0.0008602  -0.0167823  ...  0.00017693  0.01159847
  0.00638929]


In [20]:
king_queen_similarity = cosine_similarity(king_vectoer, queen_vector)
print('king-queen의 유사도', king_queen_similarity)

0.5552268369726675


In [21]:
slave_response = oepnai_client.embeddings.create(
    input="slave",
    model="text-embedding-3-large"
)

In [22]:
slave_vector = np.array(slave_response.data[0].embedding)
print(slave_vector.shape)
print(slave_vector)

(3072,)
[-0.01999537  0.00620363  0.01191717 ...  0.00094749 -0.02679118
 -0.0058524 ]


In [24]:
king_slave_similarity = cosine_similarity(king_vectoer, slave_vector)
print('king-slave의 유사도', king_slave_similarity)

king-slave의 유사도 0.2947745074537996


In [ ]:
# 한국어 문장을 벡터로 바꿔도 유사도가 비슷해야 할듯

In [26]:
kor_king_response = oepnai_client.embeddings.create(
    input="왕",
    model="text-embedding-3-large"
)
kor_king_vector = np.array(kor_king_response.data[0].embedding)
print(kor_king_vector.shape)
print(kor_king_vector)

(3072,)
[-0.00595223  0.01159333 -0.01316932 ... -0.00357134  0.01323696
 -0.00083999]


In [28]:
kor_queen_response = oepnai_client.embeddings.create(
    input="여왕",
    model="text-embedding-3-large"
)
kor_queen_vector = np.array(kor_queen_response.data[0].embedding)
print(kor_queen_vector.shape)
print(kor_queen_vector)

(3072,)
[-0.01307151 -0.00921458 -0.00532257 ... -0.00482468 -0.00204418
  0.02035061]


In [29]:
kor_king_queen_similarity = cosine_similarity(kor_king_vector, kor_queen_vector)
print('왕-여왕의 유사도', kor_king_queen_similarity)

왕-여왕의 유사도 0.48733449549538954


In [30]:
kor_slave_response = oepnai_client.embeddings.create(
    input="거지",
    model="text-embedding-3-large"
)
kor_slave_vector = np.array(kor_slave_response.data[0].embedding)
print(kor_slave_vector.shape)
print(kor_slave_vector)

(3072,)
[-0.02400834 -0.02815736 -0.00371585 ...  0.01028707 -0.00947125
  0.03754314]


# 4. upstage의 embedding model 사용
- 영어문장과 한국어 문장의 유사도는 상호 호환이 되지 않아 계산이 제대로 되지 않는다
- https://console.upstage.ai/api-keys?api=embeddings

In [34]:
import os
 
upstage_client = OpenAI(
    api_key=os.getenv("UPSTAGE_KEY"),
    base_url="https://api.upstage.ai/v1"
)

In [39]:
upstage_king_response = upstage_client.embeddings.create(
    input="king",
    model="embedding-query"
)

In [40]:
upstage_king_vector = np.array(upstage_king_response.data[0].embedding)
print(upstage_king_vector.shape)
print(upstage_king_vector)

(4096,)
[-0.01187134 -0.02058411 -0.00674438 ... -0.01082611  0.00244713
  0.01517487]


In [36]:
upstage_queen_response = upstage_client.embeddings.create(
    input="queen",
    model="embedding-query"
)
upstage_queen_vector = np.array(upstage_queen_response.data[0].embedding)
print(upstage_queen_vector.shape)
print(upstage_queen_vector)

(4096,)
[-0.0016222  -0.00952148 -0.00471878 ...  0.00985718 -0.00732803
  0.0259552 ]


In [41]:
cosine_similarity(upstage_king_vector, upstage_queen_vector)

np.float64(0.6277983746920601)

In [42]:
up_kor_king_response = upstage_client.embeddings.create(
    input="왕",
    model="embedding-query"
)
up_kor_king_vector = np.array(up_kor_king_response.data[0].embedding)
print(up_kor_king_vector.shape)
print(up_kor_king_vector)

up_kor_queen_response = upstage_client.embeddings.create(
    input="여왕",
    model="embedding-query"
)
up_kor_queen_vector = np.array(up_kor_queen_response.data[0].embedding)
print(up_kor_queen_vector.shape)
print(up_kor_queen_vector)


(4096,)
[-0.01207733 -0.0224762  -0.01322937 ... -0.00020826  0.00362587
  0.01420593]
(4096,)
[ 0.00019884 -0.00331497 -0.0114212  ...  0.00339127 -0.0071907
  0.01702881]


TypeError: unsupported operand type(s) for *: 'float' and 'CreateEmbeddingResponse'

In [43]:
cosine_similarity(up_kor_king_vector, up_kor_queen_vector)

np.float64(0.6811247524496584)